# 01 — Linear Regression: Dự báo giá đóng cửa ngày tiếp theo

**Mục tiêu:** Dự báo `close_next_day` (giá đóng cửa ngày N+1) cho từng coin dựa trên các chỉ số OHLCV + kỹ thuật của ngày N.

**Nguồn dữ liệu:** `gold.fact_market_daily` — PostgreSQL Gold layer  
**Model:** `sklearn.linear_model.LinearRegression`  
**Reuse:** Adapt từ `datamining_analysis.py` phần 2 — đổi `Ticker` → `symbol`, đổi nguồn từ CSV → PostgreSQL

---
**Các bước:**
1. Load dữ liệu từ PostgreSQL
2. Feature engineering (tạo `close_next_day`)
3. Train/Test split theo thời gian (80/20)
4. Fit Linear Regression
5. Đánh giá: MAE, RMSE, R²
6. Visualize: Actual vs Predicted, Residuals


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sqlalchemy import create_engine, text
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

print('Libraries loaded OK')

In [ ]:
# ── Database connection ───────────────────────────────────────────────────────
# Đọc config từ .env nếu có, fallback về default
def load_env(path: str = '../.env') -> dict:
    env = {}
    if not os.path.exists(path):
        path = '../.env.example'
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    k, v = line.split('=', 1)
                    env[k.strip()] = v.strip()
    return env

env = load_env()
PG_HOST = env.get('PG_HOST', '127.0.0.1')
PG_PORT = env.get('PG_PORT', '5432')
PG_DB   = env.get('PG_DATABASE', 'crypto_dw_etl')
PG_USER = env.get('PG_USER', 'crypto_etl')
PG_PASS = env.get('PG_PASSWORD', 'crypto_etl')

DSN = f'postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}'
engine = create_engine(DSN)

# Test connection
with engine.connect() as conn:
    rows = conn.execute(text('SELECT COUNT(*) FROM gold.fact_market_daily')).fetchone()
    print(f'Connected OK — gold.fact_market_daily: {rows[0]:,} rows')

In [ ]:
# ── Load data from Gold layer ─────────────────────────────────────────────────
SQL = """
SELECT
    d.full_date,
    c.symbol,
    c.full_name,
    cat.category_name,
    f.open,
    f.high,
    f.low,
    f.close,
    f.volume_base,
    f.volume_usd,
    f.return_pct,
    f.log_return,
    f.volatility,
    f.average_price,
    f.z_score_close,
    f.is_anomaly,
    f.market_dominance_pct,
    f.regime_label
FROM gold.fact_market_daily f
JOIN gold.dim_date d ON d.date_id = f.date_id
JOIN gold.dim_coin c ON c.coin_id = f.coin_id
JOIN gold.dim_category cat ON cat.category_id = f.category_id
ORDER BY c.symbol, d.full_date
"""

df_raw = pd.read_sql(SQL, engine, parse_dates=['full_date'])
print(f'Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols')
print(f'Coins : {sorted(df_raw["symbol"].unique())}')
print(f'Period: {df_raw["full_date"].min().date()} → {df_raw["full_date"].max().date()}')
df_raw.head(3)

## 1. Feature Engineering

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
# Target: close_next_day = close của ngày N+1 (shift -1 theo từng coin)
FEATURES = [
    'open', 'high', 'low', 'close',
    'volume_usd', 'return_pct', 'log_return',
    'volatility', 'average_price', 'z_score_close',
    'market_dominance_pct'
]
TARGET = 'close_next_day'

def make_features(group: pd.DataFrame) -> pd.DataFrame:
    g = group.sort_values('full_date').copy()
    g[TARGET] = g['close'].shift(-1)
    # Lag features (ngày hôm qua)
    g['close_lag1']      = g['close'].shift(1)
    g['return_pct_lag1'] = g['return_pct'].shift(1)
    g['volume_usd_lag1'] = g['volume_usd'].shift(1)
    # Rolling 7-day mean close
    g['close_ma7'] = g['close'].rolling(7, min_periods=1).mean()
    return g

df = (
    df_raw
    .groupby('symbol', group_keys=False)
    .apply(make_features)
    .dropna(subset=FEATURES + [TARGET])
    .reset_index(drop=True)
)

FEATURES_EXT = FEATURES + ['close_lag1', 'return_pct_lag1', 'volume_usd_lag1', 'close_ma7']

print(f'After feature engineering: {df.shape[0]:,} rows')
df[['full_date', 'symbol', 'close', TARGET] + FEATURES_EXT[:4]].head(5)

## 2. Train / Test Split (theo thời gian — 80% train, 20% test)

In [ ]:
# ── Time-based split ──────────────────────────────────────────────────────────
# Dùng toàn bộ coin cùng nhau, split theo ngày (không shuffle)
sorted_dates = df['full_date'].sort_values().unique()
split_idx    = int(len(sorted_dates) * 0.8)
split_date   = sorted_dates[split_idx]

df_train = df[df['full_date'] <  split_date].copy()
df_test  = df[df['full_date'] >= split_date].copy()

X_train = df_train[FEATURES_EXT].values
y_train = df_train[TARGET].values
X_test  = df_test[FEATURES_EXT].values
y_test  = df_test[TARGET].values

print(f'Split date : {pd.Timestamp(split_date).date()}')
print(f'Train rows : {len(df_train):,}  |  Test rows: {len(df_test):,}')

## 3. Fit Linear Regression

In [ ]:
# ── Scale + fit ───────────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_s, y_train)

y_pred_train = model.predict(X_train_s)
y_pred_test  = model.predict(X_test_s)

print('Model fitted!')
print(f'Intercept : {model.intercept_:.4f}')
coef_df = pd.DataFrame({'feature': FEATURES_EXT, 'coef': model.coef_}).sort_values('coef', key=abs, ascending=False)
print('\nTop-5 features by |coefficient|:')
print(coef_df.head().to_string(index=False))

## 4. Đánh giá mô hình

In [ ]:
# ── Evaluation metrics ────────────────────────────────────────────────────────
def evaluate(y_true, y_pred, split_name: str) -> dict:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    print(f'  [{split_name}]  MAE={mae:,.2f}  RMSE={rmse:,.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'split': split_name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'R2': round(r2, 4), 'MAPE_pct': round(mape, 2)}

print('Overall performance (all coins combined):')
metrics_train = evaluate(y_train, y_pred_train, 'Train')
metrics_test  = evaluate(y_test,  y_pred_test,  'Test ')
metrics_df = pd.DataFrame([metrics_train, metrics_test])
metrics_df

In [ ]:
# ── Per-coin evaluation on test set ──────────────────────────────────────────
per_coin = []
for sym, grp in df_test.groupby('symbol'):
    X_s = scaler.transform(grp[FEATURES_EXT].values)
    y_p = model.predict(X_s)
    y_t = grp[TARGET].values
    mae  = mean_absolute_error(y_t, y_p)
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    r2   = r2_score(y_t, y_p)
    mape = np.mean(np.abs((y_t - y_p) / np.where(y_t == 0, 1, y_t))) * 100
    per_coin.append({'symbol': sym, 'MAE': round(mae, 2),
                     'RMSE': round(rmse, 2), 'R2': round(r2, 4),
                     'MAPE_%': round(mape, 2)})

per_coin_df = pd.DataFrame(per_coin).sort_values('R2', ascending=False)
print('Per-coin test metrics:')
per_coin_df

## 5. Visualization

In [ ]:
# ── Plot 1: Actual vs Predicted — chọn coin BTC làm ví dụ ────────────────────
COIN = 'BTC'
df_coin_test = df_test[df_test['symbol'] == COIN].copy().sort_values('full_date')
X_coin_s = scaler.transform(df_coin_test[FEATURES_EXT].values)
df_coin_test['predicted'] = model.predict(X_coin_s)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_coin_test['full_date'], y=df_coin_test[TARGET],
    mode='lines', name='Actual close (next day)',
    line=dict(color='#00d4ff', width=2)
))
fig.add_trace(go.Scatter(
    x=df_coin_test['full_date'], y=df_coin_test['predicted'],
    mode='lines', name='Predicted close (next day)',
    line=dict(color='#ff6b6b', width=2, dash='dash')
))
fig.update_layout(
    title=f'{COIN} — Actual vs Predicted Close (Next Day) [Test Set]',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_dark', height=450,
    legend=dict(orientation='h', y=1.02)
)
fig.show()

In [ ]:
# ── Plot 2: Scatter Actual vs Predicted (all coins, test set) ─────────────────
df_test_viz = df_test.copy()
df_test_viz['predicted'] = y_pred_test

fig2 = px.scatter(
    df_test_viz, x=TARGET, y='predicted',
    color='symbol', opacity=0.6,
    labels={TARGET: 'Actual Close Next Day (USD)', 'predicted': 'Predicted (USD)'},
    title='Actual vs Predicted — All Coins (Test Set)',
    template='plotly_dark', height=500
)
# Perfect prediction line
vmin = df_test_viz[TARGET].min()
vmax = df_test_viz[TARGET].max()
fig2.add_trace(go.Scatter(
    x=[vmin, vmax], y=[vmin, vmax],
    mode='lines', name='Perfect fit',
    line=dict(color='white', dash='dot', width=1)
))
fig2.show()

In [ ]:
# ── Plot 3: Residuals ─────────────────────────────────────────────────────────
df_test_viz['residual'] = df_test_viz[TARGET] - df_test_viz['predicted']

fig3 = make_subplots(rows=1, cols=2,
    subplot_titles=['Residuals over Time (BTC)', 'Residual Distribution (All Coins)'])

btc_viz = df_test_viz[df_test_viz['symbol'] == COIN].sort_values('full_date')
fig3.add_trace(go.Scatter(
    x=btc_viz['full_date'], y=btc_viz['residual'],
    mode='lines+markers', name='Residual',
    marker=dict(size=4), line=dict(color='#ffd166')
), row=1, col=1)
fig3.add_hline(y=0, line_color='white', line_dash='dot', row=1, col=1)

fig3.add_trace(go.Histogram(
    x=df_test_viz['residual'], nbinsx=50,
    name='Residual hist', marker_color='#06d6a0'
), row=1, col=2)

fig3.update_layout(template='plotly_dark', height=400,
    title='Residual Analysis — Linear Regression')
fig3.show()

In [ ]:
# ── Plot 4: Feature coefficients ─────────────────────────────────────────────
fig4 = px.bar(
    coef_df, x='coef', y='feature', orientation='h',
    color='coef', color_continuous_scale='RdBu_r',
    title='Linear Regression — Feature Coefficients (Standardized)',
    template='plotly_dark', height=450
)
fig4.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False)
fig4.show()

In [ ]:
# ── Plot 5: Per-coin R² bar chart ─────────────────────────────────────────────
fig5 = px.bar(
    per_coin_df, x='symbol', y='R2',
    color='R2', color_continuous_scale='Viridis',
    text='R2', title='R² Score per Coin (Test Set)',
    template='plotly_dark', height=400,
    labels={'R2': 'R² Score'}
)
fig5.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig5.update_layout(coloraxis_showscale=False)
fig5.show()

## 6. Kết luận

| Metric | Train | Test |
|--------|-------|------|
| MAE | — | — |
| RMSE | — | — |
| R² | — | — |
| MAPE | — | — |

> **Nhận xét:**  
> - Linear Regression hoạt động tốt cho các coin có giá ổn định (BTC, ETH) do quan hệ gần tuyến tính giữa `close` hôm nay và `close_next_day`.  
> - MAPE và RMSE cao hơn với các Meme coin (DOGE) do biến động phi tuyến lớn.  
> - Feature quan trọng nhất: `close` (ngày hiện tại) và `close_ma7` (trung bình 7 ngày).  
> - Mô hình không nắm bắt được các cú shock thị trường (anomalies) — xem notebook `03_anomaly.ipynb`.
